# Tourist attractions ETL (HDB_Data → analytics database)

Reads **`tourist_attractions`** from **`HDB_Data`**, applies transformations in pandas, and writes **`tourist_attractions_transformed`** to a **separate MySQL database** (e.g. `tourist_analytics`).

**Prerequisites**
- MySQL running; ingest DAG has populated `HDB_Data.tourist_attractions`.
- For **`df.to_sql`** to MySQL, install **SQLAlchemy ≥ 2.0** (e.g. `pip install 'sqlalchemy>=2.0.36'`). Pandas 2.2+ ignores SQLAlchemy 1.4 and misroutes to SQLite otherwise.
- Create the target database and grant your user (run once as admin):

```sql
CREATE DATABASE IF NOT EXISTS transformed_data;
GRANT ALL PRIVILEGES ON HDB_Data.* TO 'airflow_user'@'localhost';
GRANT ALL PRIVILEGES ON transformed_data.* TO 'airflow_user'@'localhost';
FLUSH PRIVILEGES;
```

Edit the **configuration** cell below (user, password, host, database names).

In [1]:
import pandas as pd
import pymysql
from sqlalchemy import create_engine
import mysql.connector
import sqlalchemy

In [2]:
transformed_data = mysql.connector.connect(
	host='localhost',
	user='airflow_user',
	passwd='password',
	database='transformed_data'
)

cursor = transformed_data.cursor()
cursor.execute('DROP TABLE IF EXISTS tourist_attractions_transformed;')

transformed_data.commit()
transformed_data.close()

## Extract
Load the full source table into a DataFrame.

In [4]:
str_sql = '''
SELECT *
FROM tourist_attractions 
'''
df = pd.read_sql(sql=str_sql, con=HDB_Data)
df.head()

/tmp/ipykernel_45444/2401354839.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql=str_sql, con=HDB_Data)


,objectid_1,url_path,image_path,image_alt_text,photocredits,pagetitle,lastmodified,latitude,longtitude,address,postalcode,overview,external_link,meta_description,opening_hours,inc_crc,fmel_upd_d,longitude,_fp
0,1001,www.yoursingapore.com/en/see-do-singapore/arts...,www.yoursingapore.com/content/dam/desktop/glob...,None,Â©Darren Soh/National Gallery,National Gallery Singapore,2015-10-23T18:03:40.939+08:00,1.29,103.851356,1 St Andrew's Road,None,Take in the regionâ€™s newest and largest muse...,http://www.nationalgallery.sg,Take in the regionâ€™s newest and largest muse...,"Effective from 24 November 2015, Sunday to Thu...",8A41138FE0684CA7,20180619175243,103.8513559997,6aa9662ee283226c31cc0312a30e227a578a1eac019875...
1,1002,www.yoursingapore.com/en/see-do-singapore/cult...,None,None,None,Sultan Mosque (Masjid Sultan) Singapore,2015-11-02T10:27:45.190+08:00,1.302,103.859171,3 Muscat Street,None,"Also known as Masjid Sultan, the impressive Su...",http://www.sultanmosque.org.sg,The impressive Sultan Mosque (also known as Ma...,Monday to Sunday:9.30am â€“ 12pm and 2pm â€“ 4...,A3218DB77F17BFC6,20180619175243,103.8591710003,ffd74d4e38e7d2dc184bae1eb1a05d738121937586554a...
2,1003,www.yoursingapore.com/en/see-do-singapore/cult...,www.yoursingapore.com/content/dam/desktop/glob...,Sri Mariamman Temple's ornate and elaborate de...,Joel Chua DY,Sri Mariamman Temple: Hindu Temple in Singapore,2015-11-02T10:31:58.848+08:00,1.282,103.84538,244 South Bridge Road,None,"Located in Chinatown, the Sri Mariamman Temple...",http://heb.gov.sg/our-subsidiaries/temples/sri...,"Sri Mariamman Temple, located in Chinatown, is...","Daily from 7am â€“ 12pm, and 6pm â€“ 9pm",F66A0BB9C79777F1,20180619175243,103.8453800002,d1c46d838ff048490c953cd92aa13a47cd4355513bf8fe...
3,1004,www.yoursingapore.com/en/see-do-singapore/cult...,www.yoursingapore.com/content/dam/desktop/glob...,The Armenian Church in Singapore is a 19th-cen...,Joel Chua DY,Armenian Church in Singapore,2015-11-02T10:34:55.710+08:00,1.293,103.84966,60 Hill Street,None,The oldest Christian church in Singapore is an...,http://www.armeniansinasia.org/,The Armenian Church is the oldest Christian ch...,"Daily, 9am â€“6pm",4C9C762C0E30783B,20180619175243,103.8496600003,7e3894409995e22e9d99216892827daa195c3439211ff9...
4,1005,www.yoursingapore.com/en/see-do-singapore/arch...,www.yoursingapore.com/content/dam/desktop/glob...,"The charm of CHIJMES, Singapore harks back to ...",None,CHIJMES Singapore,2015-11-02T10:13:35.220+08:00,1.295,103.85168,30 Victoria Street,None,Whether functioning as a school or a lifestyle...,http://chijmes.com.sg/,Who would have thought a former school would m...,"Daily, 24 hours,Opening hours of businesses in...",5A0FF621153ABB8F,20180619175243,103.8516800002,3577d57a8545461c36da70768190cdf235a2e8daf6845a...


## Clean

Normalize raw strings and row keys before coordinate and column transforms.

In [5]:
lon_col = "longitude" if "longitude" in df.columns else "longtitude"

df["lat_wgs84"] = pd.to_numeric(df["latitude"], errors="coerce")
df["lon_wgs84"] = pd.to_numeric(df[lon_col], errors="coerce")
# Optional: keep only rows with valid coordinates
df = df.dropna(subset=["lat_wgs84", "lon_wgs84"])
# Singapore rough bounds (filters obvious outliers)
sg = (df["lat_wgs84"].between(1.15, 1.50)) & (df["lon_wgs84"].between(103.55, 104.20))
df = df.loc[sg]

df = df.drop(
	columns=[c for c in ("latitude", "longitude", "longtitude") if c in df.columns],
	errors="ignore",
)
attr_df = df.copy()

In [6]:
df.to_sql(
	name='tourist_attractions_transformed',
	con=transformed_data,
	if_exists='replace'
)
transformed_data.commit()

In [7]:
str_sql = '''
SELECT *
FROM hdb
'''
df = pd.read_sql(sql=str_sql, con=HDB_Data)
df.head()

/tmp/ipykernel_45444/630132218.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql=str_sql, con=HDB_Data)


,unnamed:_0,blk_no,street,max_floor_lvl,year_completed,residential,commercial,market_hawker,miscellaneous,multistorey_carpark,...,addr,postal,subzone_no,subzone_n,subzone_c,pln_area_n,pln_area_c,region_n,region_c,_fp
0,0,1,BEACH RD,16,1970,Y,Y,N,N,N,...,1 BEACH ROAD RAFFLES HOTEL SINGAPORE 189673,189673,2,CITY HALL,DTSZ02,DOWNTOWN CORE,DT,CENTRAL REGION,CR,81e67e7fefd009a2ebbd2f2268f74da5f40483805022fb...
1,1,1,BEDOK STH AVE 1,14,1975,Y,N,N,Y,N,...,1 BEDOK SOUTH AVENUE 1 SINGAPORE 460001,460001,6,BEDOK SOUTH,BDSZ06,BEDOK,BD,EAST REGION,ER,02553d941e0e711e9d72bbf8ad8cd4fbc71c33ad3a49c7...
2,2,1,CANTONMENT RD,2,2010,N,Y,N,N,N,...,1 CANTONMENT ROAD PINNACLE @ DUXTON SINGAPORE ...,080001,3,CHINATOWN,OTSZ03,OUTRAM,OT,CENTRAL REGION,CR,5e1c2fa09d728ea8a31983ba4d8fc5c7bc118787869f07...
3,3,1,CHAI CHEE RD,15,1982,Y,N,N,N,N,...,1 CHAI CHEE ROAD PING YI GARDENS SINGAPORE 461001,461001,3,KEMBANGAN,BDSZ03,BEDOK,BD,EAST REGION,ER,16243deb5666f52c3fdd3dd60ad7ab786ad179701f86e6...
4,4,1,CHANGI VILLAGE RD,4,1975,Y,Y,N,N,N,...,1 CHANGI VILLAGE ROAD OCBC CHANGI VILLAGE ROAD...,500001,1,CHANGI POINT,CHSZ01,CHANGI,CH,EAST REGION,ER,97f514fb11d2c1cfa34565a941cb99caf95cd79d631ecd...


In [8]:
print(df.columns.tolist())


['unnamed:_0', 'blk_no', 'street', 'max_floor_lvl', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark', 'precinct_pavilion', 'bldg_contract_town', 'total_dwelling_units', '1room_sold', '2room_sold', '3room_sold', '4room_sold', '5room_sold', 'exec_sold', 'multigen_sold', 'studio_apartment_sold', '1room_rental', '2room_rental', '3room_rental', 'other_room_rental', 'lat', 'lng', 'building', 'addr', 'postal', 'subzone_no', 'subzone_n', 'subzone_c', 'pln_area_n', 'pln_area_c', 'region_n', 'region_c', '_fp']


In [9]:
df["lat_wgs84"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon_wgs84"] = pd.to_numeric(df['lng'], errors="coerce")
# Optional: keep only rows with valid coordinates
df = df.dropna(subset=["lat_wgs84", "lon_wgs84"])
# Singapore rough bounds (filters obvious outliers)
sg = (df["lat_wgs84"].between(1.15, 1.50)) & (df["lon_wgs84"].between(103.55, 104.20))
df = df.loc[sg]

df = df.drop(
	columns=[c for c in ("lng", "lat") if c in df.columns],
	errors="ignore",
)
hdb_df=df.copy()

In [10]:
import numpy as np
from sklearn.neighbors import BallTree

# --- set your column names ---
HDB_LAT, HDB_LON = "lat_wgs84", "lon_wgs84"   # or "latitude", "longitude"
ATTR_LAT, ATTR_LON = "lat_wgs84", "lon_wgs84"  # tourist_attractions dataframe

# E.g. hdb_df = resale / flats table; attr_df = tourist_attractions (after cleaning)
# Drop rows with missing coords first:
hdb = hdb_df.dropna(subset=[HDB_LAT, HDB_LON]).copy()
attr = attr_df.dropna(subset=[ATTR_LAT, ATTR_LON]).copy()

# Radians, (lat, lon) order for sklearn's haversine
attr_xy = np.radians(attr[[ATTR_LAT, ATTR_LON]].to_numpy())
hdb_xy = np.radians(hdb[[HDB_LAT, HDB_LON]].to_numpy())

tree = BallTree(attr_xy, metric="haversine")
dist_rad, _ = tree.query(hdb_xy, k=1)

R_EARTH_M = 6_371_000
hdb["distance_to_attraction"] = dist_rad.ravel() * R_EARTH_M

# If you need distances on the original hdb_df index (with NaN rows):
hdb_df = hdb_df.copy()
hdb_df["distance_to_attraction"] = np.nan
hdb_df.loc[hdb.index, "distance_to_attraction"] = hdb["distance_to_attraction"]

In [11]:
print(hdb_df.columns.tolist())

['unnamed:_0', 'blk_no', 'street', 'max_floor_lvl', 'year_completed', 'residential', 'commercial', 'market_hawker', 'miscellaneous', 'multistorey_carpark', 'precinct_pavilion', 'bldg_contract_town', 'total_dwelling_units', '1room_sold', '2room_sold', '3room_sold', '4room_sold', '5room_sold', 'exec_sold', 'multigen_sold', 'studio_apartment_sold', '1room_rental', '2room_rental', '3room_rental', 'other_room_rental', 'building', 'addr', 'postal', 'subzone_no', 'subzone_n', 'subzone_c', 'pln_area_n', 'pln_area_c', 'region_n', 'region_c', '_fp', 'lat_wgs84', 'lon_wgs84', 'distance_to_attraction']


In [13]:
hdb_df.to_sql(
	name='hdb_transformed',
	con=transformed_data,
	if_exists='replace',
	index=False,
)

12442